Lab 02 — VDF time with Chronos (Module 2).

Concept: Wesolowski VDF proves sequential work; verify without redoing it.
Run:  python labs/lab02_chronos_vdf.py
Exercise: python labs/run_exercises.py --module m2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexar76/aimarket-courses/blob/main/verifiable-randomness-course/notebooks/{})


In [ ]:
# Setup — run this cell once per session
# Live oracle sandbox (clones alexar76/oracles) — run once per Colab session
import os
import subprocess
import sys

REPO = "https://github.com/alexar76/aimarket-courses.git"
DEST = "/content/aimarket-courses"

if not os.path.isdir(DEST):
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, DEST], check=True)
WORKDIR = os.path.join(DEST, "verifiable-randomness-course")
os.chdir(WORKDIR)

# Live oracle sandbox — clone the AIMarket oracles this course calls, then install them.
if not os.path.isdir("_deps/oracles"):
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/alexar76/oracles.git", "_deps/oracles"], check=True)
for _pkg in ["core", "oracles/chronos", "oracles/sortes", "oracles/aestus", "oracles/platon/backend"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"_deps/oracles/{_pkg}"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[oracles,dev]"], check=True)
os.environ.setdefault("COURSE_LANG", "en")  # change to ru or es


In [ ]:
"""Lab 02 — VDF time with Chronos (Module 2).

Concept: Wesolowski VDF proves sequential work; verify without redoing it.
Run:  python labs/lab02_chronos_vdf.py
Exercise: python labs/run_exercises.py --module m2
"""



from courselib.i18n import get_translator
from courselib.randomness import vdf_eval
from courselib.trace import Trace


def main() -> None:
    t = get_translator()
    trace = Trace()
    print(f"== {t('modules.m2.title')} ==")

    seed = t("labs.lab02.seed")
    out = vdf_eval(seed, difficulty=700)
    trace.log("vdf_eval", difficulty=out["difficulty"], valid=out["valid"])

    print(f"{t('ui.proof')}: pi={str(out['proof']['pi'])[:24]}… l={out['proof']['l']}")
    print(f"{t('ui.verify')}: {out['valid']}")

    # Tamper demo
    ensure = __import__("courselib.oracle_paths", fromlist=["ensure_oracle"]).ensure_oracle
    ensure("chronos")
    from chronos import vdf

    tampered = vdf.verify(out["g"], out["y"] + 1, out["difficulty"], out["proof"]["pi"], out["proof"]["l"])
    trace.log("tamper_check", valid=tampered)
    print(f"{t('labs.lab02.tampered')}: {tampered}")

    print(f"\n{t('ui.trace')} ({len(trace)} {t('ui.events')}):")
    for e in trace.events:
        print(" ", e)

    print(f"\n--- {t('exercises.heading')} ---")
    print(t("exercises.lab02_hint"))

main()
